# Radiology AI — Phase 2: VinDr-CXR detection (YOLOv8s, 1024px)

Repo: `DonatoDiaz/radiology-ai` · 14 finding classes · data **3.7 GB**.

**Требования:** Google Drive с файлом `vindr-cxr-coco.zip` (весь датасет, ~3.6 GB).
Положить его в папку по пути, указанному в ячейке ниже (`ZIP_IN_DRIVE`), затем Run all.

---

**Как положить zip в Drive:** ✓ `drive.google.com` → `My Drive` → создать папку `radiology_data` → перетащить файл `vindr-cxr-coco.zip` (загружается ~15–60 мин при 10–50 МБ/с).

In [ ]:
import torch, subprocess, os
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print('VRAM GB:', torch.cuda.get_device_properties(0).total_memory / 1e9)
print('Python:', subprocess.run(['python','--version'], capture_output=True, text=True).stdout.strip())
print('RAM GB:', round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9,1))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Настройки

Путь к zip в вашем Drive, имя прогона, число эпох, разрешение, batch.

In [ ]:
ZIP_IN_DRIVE = '/content/drive/MyDrive/radiology_data/vindr-cxr-coco.zip'
RUN_NAME = 'det_v1_b0_1024'      # имя папки результатов
EPOCHS   = 40                    # patience остановится раньше при отсутствии прогресса
IMG_SIZE = 1024                  # 512 не различает мелкие bbox (медиана 0.2% площади)
BATCH    = 16                    # 16 влезает в 15 GB T4; уменьшите до 8 при OOM
OUT      = '/content/work'

In [ ]:
import os, zipfile, shutil
DATA = os.path.join(OUT, 'data')
os.makedirs(OUT, exist_ok=True)
print('zip exists:', os.path.exists(ZIP_IN_DRIVE))
if os.path.exists(os.path.join(DATA, 'images/train')):
    print('data уже распаковано, пропускаю')
else:
    os.makedirs(DATA, exist_ok=True)
    with zipfile.ZipFile(ZIP_IN_DRIVE) as z:
        z.extractall(DATA)
    print('распаковано в', DATA)

In [ ]:
# найти корень датасета (папка с annotations/instances_train.json)
import glob
root = None
for cand in glob.glob(DATA + '/**/annotations/instances_train.json', recursive=True):
    root = os.path.dirname(os.path.dirname(cand))
    break
assert root, 'instances_train.json не найден — проверьте структуру zip'
print('датасет:', root)
import json
d = json.load(open(os.path.join(root, 'annotations/instances_train.json')))
print('categories:', [c['name'] for c in d['categories']])
print('train images:', len(d['images']), 'anns:', len(d['annotations']))

## COCO → YOLO

Конвертируем аннотации в формат YOLO (`cls cx cy w h`, нормализованные) в папки `labels/{train,val}`.

In [ ]:
import json
from pathlib import Path
root = Path(root)
for split in ['train', 'val']:
    ann = json.loads((root / f'annotations/instances_{split}.json').read_text())
    cat2cls = {c['id']: i for i, c in enumerate(ann['categories'])}
    img2im = {im['id']: im for im in ann['images']}
    by_img = {}
    for a in ann['annotations']:
        by_img.setdefault(a['image_id'], []).append(a)
    out = root / f'labels/{split}'
    out.mkdir(parents=True, exist_ok=True)
    n_w = 0; n_bad = 0
    for im in ann['images']:
        W, H = im['width'], im['height']
        lines = []
        for a in by_img.get(im['id'], []):
            x, y, bw, bh = a['bbox']
            cx, cy = (x+bw/2)/W, (y+bh/2)/H
            w, h = bw/W, bh/H
            if cx<0 or cy<0 or w<=0 or h<=0:
                n_bad += 1; continue
            if cx>1 or cy>1 or cx+w/2>1 or cy+h/2>1:
                w = min(2*min(cx,1-cx), w); h = min(2*min(cy,1-cy), h)
            lines.append(f"{cat2cls[a['category_id']]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
        txt = Path(im['file_name']).with_suffix('.txt')
        (out/txt).write_text('\n'.join(lines)+('\n' if lines else ''))
        n_w += 1
    print(f'[{split}] labels: {n_w} файлов, bad={n_bad}')

In [ ]:
# конфиг YOLO с абсолютными путями
cfg = f"""
path: {root}
train: images/train
val: images/val
names:
  0: Aortic enlargement
  1: Pleural thickening
  2: Pleural effusion
  3: Cardiomegaly
  4: Lung Opacity
  5: Nodule/Mass
  6: Consolidation
  7: Pulmonary fibrosis
  8: Infiltration
  9: Atelectasis
  10: Other lesion
  11: ILD
  12: Pneumothorax
  13: Calcification
"""
cfg_path = os.path.join(OUT, 'det_vindr.yaml')
open(cfg_path, 'w').write(cfg)
print(cfg_path)

In [ ]:
%pip install -q ultralytics
import ultralytics
print('ultralytics', ultralytics.__version__)

## Обучение

YOLOv8s, imgsz 1024, batch 16, до 40 эпох с early stopping (patience 15).
Темп на T4: ~4–6 мин/эпоху → полный прогон ≤ 4 ч. Сохранение весов каждую эпоху → переживает разрыв сессии (потеряется только остановленное time-окно).

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8s.pt')  # предобученный COCO
yaml_path = cfg_path
kwargs = dict(
    data=yaml_path,
    imgsz=IMG_SIZE,
    batch=BATCH,
    epochs=EPOCHS,
    patience=15,
    project=os.path.join(OUT, 'runs'),
    name=RUN_NAME,
    exist_ok=True,
    device=0,
    workers=4,
    amp=True,
)
print('training...')
results = model.train(**kwargs)

## Метрики

mAP50 / mAP50-95 на валидации — таблица и результаты в `runs/{RUN_NAME}`.

In [ ]:
best = os.path.join(OUT, 'runs', RUN_NAME, 'weights', 'best.pt')
m = YOLO(best)
mtr = m.val(data=yaml_path, imgsz=IMG_SIZE, device=0)
print('mAP50 %.4f  mAP50-95 %.4f' % (mtr.box.map50, mtr.box.map))
for i, c in enumerate([c['name'] for c in json.load(open(root/'annotations/instances_val.json'))['categories']]):
    print(f'{c:24s} mAP50 {mtr.box.maps[i]:.3f}')

In [ ]:
# сохранить результаты и веса обратно в Drive
import shutil
dest = '/content/drive/MyDrive/radiology_data/phase2_models/'
os.makedirs(dest, exist_ok=True)
shutil.copy(best, dest + 'best_' + RUN_NAME + '.pt')
shutil.copy(os.path.join(OUT, 'runs', RUN_NAME, 'results.csv'), dest + 'results_' + RUN_NAME + '.csv')
shutil.copy(os.path.join(OUT, 'runs', RUN_NAME, 'results.png'), dest + 'results_' + RUN_NAME + '.png')
print('сохранено в', dest)
!ls -lh {dest}